# Chạy đánh giá danh sách câu hỏi `eval/results/cau-hoi01.md`

Notebook này nạp pipeline RAG (`RAG_PROFILE=server`), đọc toàn bộ 56 câu hỏi từ `eval/results/cau-hoi01.md`, truy vấn và lưu kết quả ra file định dạng tương tự `eval/results/test-basic.md`.

In [1]:
import os, sys, re
from pathlib import Path

# Đặt working directory về thư mục gốc của project
cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

# Thiết lập profile server
os.environ["RAG_PROFILE"] = "server"

from config import get_config
from index_builder import build_or_load_index
from query_engine import build_query_engine, format_sources

cfg = get_config()
cfg.streaming = False  # Chạy batch không cần streaming ra console
print(f"Profile: {cfg.name} | LLM: {cfg.llm_model_name} | Ollama: {cfg.ollama_base_url}")

/mnt/data/1_Projects/sgu-rag-llamaindex/venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Profile: server | LLM: qwen3:8b | Ollama: http://100.122.2.3:11434


In [2]:
# Khởi tạo Index và Query Engine
index = build_or_load_index(cfg)
engine = build_query_engine(index, cfg)
print("Query engine đã sẵn sàng!")

Query engine đã sẵn sàng!


In [3]:
# Đọc danh sách câu hỏi từ eval/results/cau-hoi01.md
input_file = ROOT / "eval" / "results" / "cau-hoi01.md"
questions = []

with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        m = re.match(r"^\d+\.\s*(.+)$", line.strip())
        if m:
            questions.append(m.group(1).strip())

print(f"Đã đọc {len(questions)} câu hỏi từ {input_file.name}:")
for i, q in enumerate(questions[:5], 1):
    print(f"  {i}. {q}")
print("  ...")

Đã đọc 56 câu hỏi từ cau-hoi01.md:
  1. nếu tôi được 8.4 hệ 10 thì hệ 4 được bao nhiêu điểm
  2. 3.4 điểm thì được A, B, c <d
  3. tôi học cử nhân thì cần học bao nhiêu chỉ
  4. còn kỹ sư
  5. tôi học kỹ sư thì cần học bao nhiêu chỉ
  ...


In [ ]:
# Chạy toàn bộ câu hỏi và lưu vào file Markdown
from tqdm.auto import tqdm

output_file = ROOT / "eval" / "results" / "cau-hoi01-result.md"

with open(output_file, "w", encoding="utf-8") as out:
    out.write(f"Hỏi đáp quy chế, quy định đào tạo SGU ({cfg.corpus_dir}, rerank: {'bật' if cfg.use_rerank else 'tắt'}). Gõ 'thoat' để dừng.\n\n")
    
    for i, q in enumerate(tqdm(questions, desc="Đang xử lý câu hỏi"), 1):
        resp = engine.query(q)
        ans_text = str(resp).strip()
        sources = format_sources(resp)
        
        entry = f"Câu hỏi: {q}\n{ans_text}\nNguồn: {sources}\n\n"
        out.write(entry)
        out.flush()
        print(f"[{i}/{len(questions)}] Xong: {q[:40]}...")

print(f"\nHoàn tất! Kết quả đã được lưu tại: {output_file}")

Đang xử lý câu hỏi:   0%|          | 0/56 [00:00<?, ?it/s]